In [4]:
import sys
print(sys.executable)
print(sys.version)
!{sys.executable} -m pip install opencv-python torch pandas


c:\Users\Admin\AppData\Local\Programs\Python\Python312\python.exe
3.12.0 (tags/v3.12.0:0fb18b0, Oct  2 2023, 13:03:39) [MSC v.1935 64 bit (AMD64)]

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



  Using cached opencv_python-4.13.0.90-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached numpy-2.4.1-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
Using cached opencv_python-4.13.0.90-cp37-abi3-win_amd64.whl (40.2 MB)
Using cached numpy-2.4.1-cp312-cp312-win_amd64.whl (12.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.3
    Uninstalling numpy-1.26.3:
      Successfully uninstalled numpy-1.26.3


In [1]:
import os
import cv2
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [22]:
# ------------------------
# Dataset PyTorch pour eyetracking
# ------------------------
class EyeTrackingDataset(Dataset):
    def __init__(self, csv_file, img_dir, img_size=(224,224), transform=None):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.img_size = img_size
        self.transform = transform

        # filtrer les lignes où l'image existe et est valide
        valid_indices = []
        for idx, row in self.df.iterrows():
            fid = int(row['frame_id'])
            img_path = os.path.join(img_dir, f"frame_{fid:04d}.png")
            if os.path.exists(img_path):
                img = cv2.imread(img_path)
                if img is not None and len(img.shape) == 3 and img.shape[0] > 0 and img.shape[1] > 0:
                    valid_indices.append(idx)
        self.df = self.df.iloc[valid_indices].reset_index(drop=True)

        # créer mapping frame_id -> image path
        self.img_paths = [os.path.join(img_dir, f"frame_{int(fid):04d}.png") 
                          for fid in self.df['frame_id']]

        # extraire les coordonnées
        self.coords = self.df[['x','y']].values.astype(np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        img = cv2.imread(img_path)
        if img.ndim == 2:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        elif img.shape[2] == 1:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # resize
        img = cv2.resize(img, self.img_size)

        # normalisation coords
        h, w, _ = img.shape
        x, y = self.coords[idx]
        x_norm = x / w
        y_norm = y / h
        label = torch.tensor([x_norm, y_norm], dtype=torch.float32)

        # transform PyTorch (ToTensor + normalisation)
        if self.transform:
            img = self.transform(img)
        else:
            # default transform
            img = transforms.ToTensor()(img)
            img = transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])(img)

        return img, label

In [23]:
csv_file = "C:\\Users\\Admin\\Documents\\5A\\a_learning\\AL_project\\active_learning\\data\\coords_20260121_082431.csv"

# Créer les datasets séparés
train_dataset = EyeTrackingDataset(csv_file, "data/dataset/train")
dev_dataset = EyeTrackingDataset(csv_file, "data/dataset/dev")
test_dataset = EyeTrackingDataset(csv_file, "data/dataset/test")

print(f"Train size: {len(train_dataset)}")
print(f"Dev size: {len(dev_dataset)}")
print(f"Test size: {len(test_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Vérification rapide
imgs, labels = next(iter(train_loader))
print(imgs.shape, labels.shape)  # imgs: [B,3,H,W], labels: [B,2]

Train size: 1248
Dev size: 269
Test size: 268
torch.Size([32, 3, 224, 224]) torch.Size([32, 2])


In [24]:
import torch.nn as nn

class GazeNet(nn.Module):
    def __init__(self):
        super(GazeNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 56 * 56, 512)
        self.fc2 = nn.Linear(512, 2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(-1, 64 * 56 * 56)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = GazeNet()
print(model)


# Entraînement

import torch.optim as optim

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 5  

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for imgs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, labels in dev_loader:
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {running_loss/len(train_loader):.4f}, Val Loss: {val_loss/len(dev_loader):.4f}")

print("Entraînement terminé.")

GazeNet(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=200704, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=2, bias=True)
  (relu): ReLU()
)
Epoch 1/5, Train Loss: 18.2198, Val Loss: 0.5073
Epoch 2/5, Train Loss: 0.5367, Val Loss: 0.3761
Epoch 3/5, Train Loss: 0.3686, Val Loss: 0.3221
Epoch 4/5, Train Loss: 0.3009, Val Loss: 0.2998
Epoch 5/5, Train Loss: 0.2214, Val Loss: 0.1681
Entraînement terminé.


In [25]:
# Sauvegarder le modèle
torch.save(model.state_dict(), 'gaze_model2.pth')
print("Modèle sauvegardé.")

Modèle sauvegardé.


In [26]:
# Évaluation sur le test set

model.eval()
test_loss = 0.0
with torch.no_grad():
    for imgs, labels in test_loader:
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

print(f"Test Loss: {test_loss/len(test_loader):.4f}")

Test Loss: 0.3548


In [27]:
# Comparaison avec un modèle aléatoire (non entraîné)

random_model = GazeNet()  

random_model.eval()
random_test_loss = 0.0
with torch.no_grad():
    for imgs, labels in test_loader:
        outputs = random_model(imgs)
        loss = criterion(outputs, labels)
        random_test_loss += loss.item()

print(f"Test Loss du modèle aléatoire: {random_test_loss/len(test_loader):.4f}")
print(f"Test Loss du modèle entraîné: {test_loss/len(test_loader):.4f}")
print(f"Amélioration: {((random_test_loss - test_loss) / random_test_loss * 100):.2f}%")

Test Loss du modèle aléatoire: 6.3385
Test Loss du modèle entraîné: 0.3548
Amélioration: 94.40%
